In [21]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    MemorySearchOptions,
    MemoryStoreDefaultDefinition,
    MemoryStoreDefaultOptions,
    ResponsesAssistantMessageItemParam,
    ResponsesUserMessageItemParam,
)
from azure.identity import DefaultAzureCredential

In [22]:
MEMORY_STORE_NAME = "2026-02-02_1411"
SCOPE = "user"

In [23]:
# Initialize the client
client = AIProjectClient(
    # endpoint="https://docimage-dev-foundry-wus.services.ai.azure.com/api/projects/docimage-dev-foundry-wus",
    endpoint="https://customvision-dev-aoai.services.ai.azure.com/api/projects/customvision-dev-aoai-project",
    credential=DefaultAzureCredential()
)

# Create memory store
definition = MemoryStoreDefaultDefinition(
    chat_model="gpt-4.1-001",  # Your chat model deployment name
    embedding_model="text-embedding-3-small-001",  # Your embedding model deployment name
    options=MemoryStoreDefaultOptions(user_profile_enabled=True, chat_summary_enabled=True)
)

In [15]:
# List all memory stores

for store in client.memory_stores.list():
    print(f"Memory Store: {store.name}")

Memory Store: 2026-02-05_2015
Memory Store: 2026-02-05_2010
Memory Store: 2026-02-05_2005
Memory Store: 2026-02-05_2001
Memory Store: 2026-02-05_1957
Memory Store: 2026-02-05_1729
Memory Store: 2026-02-05_1725
Memory Store: 2026-02-05_1720
Memory Store: 2026-02-02_1413
Memory Store: 2026-02-02_1411
Memory Store: my_memory_store_ny


In [16]:
# check if the memory store already exists
existing_store = None
for store in client.memory_stores.list():
    if store.name == MEMORY_STORE_NAME:
        existing_store = store
        print(f"Memory store '{MEMORY_STORE_NAME}' already exists.")
        break

Memory store '2026-02-02_1411' already exists.


In [17]:
# # Delete the entire memory store
# delete_response = client.memory_stores.delete("my_memory_store_ny")
# print(f"Deleted memory store: {delete_response.deleted}")

In [18]:
# # Create a new memory store

# memory_store = client.memory_stores.create(
#     name="my_memory_store_ny",
#     definition=definition,
#     description="Memory store for interior design agent",
# )

# print(f"Created memory store: {memory_store.name}")

In [19]:
# Search Static Memory

import json

scope = "user"

result = client.memory_stores.search_memories(
    name=MEMORY_STORE_NAME,
    scope=SCOPE,
    items=[],
    options=MemorySearchOptions(max_memories=100)
)

print(len(result.memories))
print(json.dumps(result.memories,
                 indent=2,
                 ensure_ascii=False,
                 default=str))

100
[
  "{'memory_item': {'memory_id': 'd462894931734c1c83fdd7bc9647e6b5', 'updated_at': 1770426940, 'scope': 'user', 'content': 'User faces challenges translating sketches into actionable tasks for the team.', 'kind': 'user_profile'}}",
  "{'memory_item': {'memory_id': '31be6732a4a74554a5b60d3d4b2bd097', 'updated_at': 1770426940, 'scope': 'user', 'content': 'User manages a newly reorganized team under tight deadlines.', 'kind': 'user_profile'}}",
  "{'memory_item': {'memory_id': 'f25dc988b5c144c0a2f0bb063e8a1eb9', 'updated_at': 1770426940, 'scope': 'user', 'content': 'User is involved in cross-functional meetings for the Azure healthcare analytics prototype.', 'kind': 'user_profile'}}",
  "{'memory_item': {'memory_id': '7926a6f9825841ea83c15b3b897d5746', 'updated_at': 1770426940, 'scope': 'user', 'content': 'User takes photos of sketches and adds summaries to a shared task board for team visibility.', 'kind': 'user_profile'}}",
  "{'memory_item': {'memory_id': 'afbc678f1da147d2963ee41

In [25]:
# Search dynamic memory with query thread


query = "Do I still eat red meat?"

query_thread = [
    ResponsesUserMessageItemParam(
        content=query,
    )
]


result = client.memory_stores.search_memories(
    name=MEMORY_STORE_NAME,
    scope=SCOPE,
    items=query_thread,
    options=MemorySearchOptions(max_memories=10)
)

print(len(result.memories))
print(json.dumps(result.memories,
                 indent=2,
                 ensure_ascii=False,
                 default=str))

10
[
  "{'memory_item': {'memory_id': '2aa145beb9694087a9b230f7da16baf6', 'updated_at': 1770430794, 'scope': 'user', 'content': \"User has hypertension, follows a heart-healthy, plant-based, low-sodium, and lactose-free diet, avoids processed foods, and has mild lactose intolerance. User and family meal plan every Sunday to avoid processed foods and include more fresh options, focusing on lactose-free alternatives; user's daughter prefers oat milk. User and their daughter require lactose-free meal options. User is actively avoiding processed foods and prefers heart-healthy, low-sodium meals for the whole family. User and their family follow a plant-based, heart-healthy diet, avoiding red meat, processed foods, and high-sodium items, and use lactose-free alternatives. User and wife prepare veggie stir-fries with very little salt and use lactose-free dairy options. User prefers plant-based, heart-friendly meals and involving their daughter in meal prep through age-appropriate tasks and i

In [37]:
result.memories

[{'memory_item': {'memory_id': '29c58004d99d4cfeb9c2f729842fcdca', 'updated_at': 1770430777, 'scope': 'user', 'content': 'The user is seeking advice on creating a flexible weekly meal plan that accommodates a busy college schedule and upcoming travel for workshops and group events. The user prefers filling lunches, especially on active days, and requires meal ideas and planning tips that are adaptable to different countries and unpredictable routines. The user is lactose intolerant or prefers oat milk, skips breakfast, and values meals that are high in fiber and protein. The assistant provided a comprehensive set of strategies, including batch prepping versatile meal components, exploring local groceries and markets in new countries, relying on portable and shelf-stable core ingredients, and focusing on quick, one-dish meals like grain bowls, hearty salads, wraps, and bento box variations. The assistant also suggested leveraging local specialties and street food with lactose-free adjus

In [ ]:
sample_threads = [[
    ResponsesUserMessageItemParam(content="I have a brown pant and a blue bottle"),
    ResponsesAssistantMessageItemParam(content="Acknowledged")],
    [
    ResponsesUserMessageItemParam(content="I don't have a brown anymore"),
    ResponsesAssistantMessageItemParam(content="Ok")
],
    [
    ResponsesUserMessageItemParam(content="I replace my blue bottle with a red one"),
    ResponsesAssistantMessageItemParam(content="Ok")
]
]

In [ ]:
# Set scope to associate the memories with

scope = "user"
update_poller = None

for i, thread in enumerate(sample_threads):
    update_poller = client.memory_stores.begin_update_memories(
        name="my_memory_store_ny",
        scope=scope,
        items=thread,  # Pass conversation items that you want to add to memory
        previous_update_id=update_poller.update_id if i > 0 else None,  # Extend from previous update ID
        update_delay=0,  # Trigger update immediately without waiting for inactivity
    )

    # Wait for the update operation to complete, but can also fire and forget
    update_result = update_poller.result()
    print(f"Updated with {len(update_result.memory_operations)} memory operations")
    for operation in update_result.memory_operations:
        print(
            f"  - Operation: {operation.kind}, Memory ID: {operation.memory_item.memory_id}, Memory Type: {operation.memory_item.kind}, Content: {operation.memory_item.content}"
        )

In [11]:
# Use memory via agent tool


# Continue from the previous Python snippets.
from azure.ai.projects.models import MemorySearchTool, PromptAgentDefinition

# Set scope to associate the memories with
# You can also use "{{$userId}}" to take the TID and OID of the request authentication header

openai_client = client.get_openai_client()

# Create memory search tool
tool = MemorySearchTool(
    memory_store_name=MEMORY_STORE_NAME,
    scope=SCOPE,
    update_delay=5,  # Wait 1 second of inactivity before updating memories
    # In a real application, set this to a higher value like 300 (5 minutes, default)
    search_options=MemorySearchOptions(max_memories=5)
)

# Create a prompt agent with memory search tool
agent = client.agents.create_version(
    agent_name="MyAgent",
    definition=PromptAgentDefinition(
        model="gpt-4.1",
        instructions="You are a helpful assistant that answers general questions",
        tools=[tool],
    )
)



In [12]:
# Create a conversation with the agent with memory tool enabled
conversation = openai_client.conversations.create()
print(f"Created conversation (id: {conversation.id})")

# Create an agent response to initial user message
response = openai_client.responses.create(
    input="I prefer dark roast coffee",
    conversation=conversation.id,
    extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
)

print(f"Response output: {response.output_text}")



Created conversation (id: conv_cdc8f42d3778a51800eRlLqSMe7ySttLYrop93KImML9CqPplZ)


BadRequestError: Error code: 400 - {'error': {'message': "Unknown parameter: 'tools[0].search_options'.", 'type': 'invalid_request_error', 'param': 'tools[0].search_options', 'code': 'unknown_parameter'}}

In [ ]:
# Create a new conversation
new_conversation = openai_client.conversations.create()
print(f"Created new conversation (id: {new_conversation.id})")

# Create an agent response with stored memories
new_response = openai_client.responses.create(
    input="Please order my usual coffee",
    conversation=new_conversation.id,
    extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
)

print(f"Response output: {new_response.output_text}")